# UZX API Testing

In [ ]:
import base64
import hashlib
import hmac
import json
import time
from enum import Enum
from typing import Dict, Any

import requests

## Configuration

In [ ]:
base_url = "https://api-v2.uzx.com"
symbol = "GGEZ1-USDT"

In [ ]:
# Accounts — uncomment the one you want to use

# Default account
api_key = "xx"
secret_key = "xx"
passphrase = "xx"

# MUTAZ account
# api_key = "xx"
# secret_key = "xx"
# passphrase = "xx"

# MOHD account
# api_key = "xx"
# secret_key = "xx"
# passphrase = "xx"

## Enums

In [ ]:
class OrderStatus(Enum):
    PENDING = 0
    PARTIALLY_FILLED = 1
    PARTIALLY_CANCELED = 2
    CANCELED = 3
    FULLY_FILLED = 4


class OrderBuyOrSell(Enum):
    BUY = 1
    SELL = 2


class OrderTypeUZX(Enum):
    MARKET_ORDER = 1
    LIMIT_GTC = 2
    LIMIT_IOC = 3
    LIMIT_FOK = 4
    LIMIT_MAKER = 5

## Helper Functions

In [ ]:
def _parse_params(params: Dict[str, Any]) -> str:
    return "?" + "&".join(f"{k}={v}" for k, v in params.items())


def _sign(method: str, path: str, query: dict = None, body: dict = None) -> Dict[str, str]:
    """Build auth headers for a V2 API request."""
    timestamp = str(int(time.time()))
    query_str = _parse_params(query) if query else ""
    body_str = json.dumps(body, separators=(",", ":")) if body else ""
    prehash = f"{timestamp}{method.upper()}{path}{query_str}{body_str}"
    signature = hmac.new(secret_key.encode(), prehash.encode(), hashlib.sha256).digest()
    return {
        "UZX-ACCESS-KEY": api_key,
        "UZX-ACCESS-SIGN": base64.b64encode(signature).decode(),
        "UZX-ACCESS-TIMESTAMP": timestamp,
        "UZX-ACCESS-PASSPHRASE": passphrase,
        "Content-Type": "application/json",
    }


def api_get(end_point: str, params: dict = None) -> dict:
    headers = _sign("GET", end_point, query=params)
    response = requests.get(base_url + end_point, headers=headers, params=params)
    response.raise_for_status()
    return response.json()


def api_post(end_point: str, data: dict) -> dict:
    headers = _sign("POST", end_point, body=data)
    response = requests.post(
        base_url + end_point,
        headers=headers,
        data=json.dumps(data, separators=(",", ":")),
    )
    response.raise_for_status()
    return response.json()


def api_put(end_point: str, data: dict) -> dict:
    headers = _sign("PUT", end_point, body=data)
    response = requests.put(
        base_url + end_point,
        headers=headers,
        data=json.dumps(data, separators=(",", ":")),
    )
    response.raise_for_status()
    return response.json()


def api_get_public(end_point: str, params: dict = None) -> dict:
    response = requests.get(base_url + end_point, params=params)
    response.raise_for_status()
    return response.json()


def pp(data):
    """Pretty print JSON."""
    print(json.dumps(data, indent=4))

## Account Balance

In [ ]:
balances = api_get("/v2/account/balances")
for asset in balances["data"]:
    print(f"Asset: {asset['coin']} , Available: {asset['available_balance']}, Freeze: {asset['frozen_balance']}")

## Active Orders

In [ ]:
active_orders = api_get("/v2/trade/orders", {"ins_type": "SPOT", "product_name": symbol})
pp(active_orders)

## Place Order

In [ ]:
order_data = {
    "product_name": symbol,
    "order_buy_or_sell": OrderBuyOrSell.SELL.value,
    "price": "0.09",
    "amount": "1692.00",
    "order_type": OrderTypeUZX.LIMIT_GTC.value,
}
result = api_post("/v2/trade/spot/order", order_data)
pp(result)

## Cancel Orders

### Cancel all active orders

In [ ]:
orders_to_cancel = api_get("/v2/trade/orders", {"ins_type": "SPOT", "product_name": symbol})

for order in orders_to_cancel["data"]:
    result = api_put("/v2/trade/cancel-order", {
        "inst_type": 1,
        "cancel_ord_type": 1,
        "order_id": order["order_id"],
    })
    pp(result)
    time.sleep(1)

### Cancel 1 order

In [ ]:
order_id = "2604110953500203344"
result = api_put("/v2/trade/cancel-order", {
    "inst_type": 1,
    "cancel_ord_type": 1,
    "order_id": order_id,
})
pp(result)

## Trade History

In [ ]:
history = api_get("/v2/trade/history/orders", {"product_name": symbol})
pp(history)

## Market Data

In [ ]:
# Products list
products = api_get_public("/v2/products", {"ins_type": "SPOT", "product_name": symbol})
print("symbols:", len(products["data"]))
pp(products)

In [ ]:
# Server time
server_time = api_get_public("/v2/time")
pp(server_time)

## Market Data (Tickers)

In [ ]:
# Single ticker
ticker = api_get_public(f"/notification/spot/{symbol}/ticker")
pp(ticker)

In [ ]:
# All spot tickers
all_tickers = api_get_public("/notification/spot/tickers")
pp(all_tickers)

## Order Book

In [ ]:
depth = api_get_public(f"/notification/spot/{symbol}/orderbook", {"interval": "step0", "depth": 20})
pp(depth)

## Generate Order Book

In [ ]:
orders = [
    # Sell orders
    {"amount": "5000.00", "price": str(0.08775), "side": OrderBuyOrSell.SELL},
    {"amount": "5000.00", "price": str(0.0925), "side": OrderBuyOrSell.SELL},
    {"amount": "5000.00", "price": str(0.092), "side": OrderBuyOrSell.SELL},
    {"amount": "5000.00", "price": str(0.0915), "side": OrderBuyOrSell.SELL},
    {"amount": "5000.00", "price": str(0.091), "side": OrderBuyOrSell.SELL},
    {"amount": "10000.0", "price": str(0.0905), "side": OrderBuyOrSell.SELL},
    {"amount": "10000.0", "price": str(0.09), "side": OrderBuyOrSell.SELL},
    {"amount": "10000.0", "price": str(0.0895), "side": OrderBuyOrSell.SELL},
    {"amount": "12000.0", "price": str(0.089), "side": OrderBuyOrSell.SELL},
    {"amount": "14000.0", "price": str(0.0885), "side": OrderBuyOrSell.SELL},
    {"amount": "16000.0", "price": str(0.088), "side": OrderBuyOrSell.SELL},
    # Buy orders
    {"amount": "5000.00", "price": str(0.08725), "side": OrderBuyOrSell.BUY},
    {"amount": "5000.00", "price": str(0.08675), "side": OrderBuyOrSell.BUY},
    {"amount": "5000.00", "price": str(0.08625), "side": OrderBuyOrSell.BUY},
    {"amount": "5000.00", "price": str(0.08575), "side": OrderBuyOrSell.BUY},
    {"amount": "5000.00", "price": str(0.08525), "side": OrderBuyOrSell.BUY},
    {"amount": "10000.0", "price": str(0.08475), "side": OrderBuyOrSell.BUY},
    {"amount": "10000.0", "price": str(0.08425), "side": OrderBuyOrSell.BUY},
    {"amount": "10000.0", "price": str(0.08375), "side": OrderBuyOrSell.BUY},
    {"amount": "12000.0", "price": str(0.08325), "side": OrderBuyOrSell.BUY},
    {"amount": "14000.0", "price": str(0.08275), "side": OrderBuyOrSell.BUY},
    {"amount": "16000.0", "price": str(0.08225), "side": OrderBuyOrSell.BUY},
]

for order in orders:
    result = api_post("/v2/trade/spot/order", {
        "product_name": symbol,
        "order_buy_or_sell": order["side"].value,
        "price": order["price"],
        "amount": order["amount"],
        "order_type": OrderTypeUZX.LIMIT_GTC.value,
    })
    pp(result)
    time.sleep(1)

---
## V1 API Testing

### V1 Configuration

In [ ]:
v1_base_url = "https://api.uzx.com"

# Accounts — uncomment the one you want to use

# Default account
v1_api_key = "xx"
v1_secret_key = "xx"

### V1 Enums

In [ ]:
class OrderDirection(Enum):
    BUY = "BUY"
    SELL = "SELL"


class OrderType(Enum):
    MARKET_PRICE = "MARKET_PRICE"
    LIMIT_PRICE = "LIMIT_PRICE"

### V1 Helper Functions

In [ ]:
def _v1_signature(timestamp: str) -> str:
    pre_hash = f"{v1_api_key}\n{v1_secret_key}\n{timestamp}"
    digest = hmac.new(v1_secret_key.encode(), pre_hash.encode(), hashlib.sha1).digest()
    return base64.b64encode(digest).decode()


def _v1_headers() -> Dict[str, str]:
    timestamp = str(int(time.time()))
    return {
        "apiKey": v1_api_key,
        "signature": _v1_signature(timestamp),
        "timestamp": timestamp,
    }


def v1_get(end_point: str, params: dict = None) -> dict:
    response = requests.get(v1_base_url + end_point, headers=_v1_headers(), params=params)
    response.raise_for_status()
    return response.json()


def v1_get_public(end_point: str, params: dict = None) -> dict:
    response = requests.get(v1_base_url + end_point, params=params)
    response.raise_for_status()
    return response.json()

### V1 Account Balance

In [ ]:
balances = v1_get("/api/asset/wallet")
for asset in balances["data"]:
    if asset["totalBalance"] == 0:
        continue
    print(f"Asset: {asset['coin']['unit']} , Available: {asset['balance']}, Freeze: {asset['frozenBalance']}")

### V1 Recent Transactions

In [ ]:
trades = v1_get("/api/latest-trade", {"symbol": "GGEZ1/USDT", "size": 5})
pp(trades)

### V1 Place Order

In [ ]:
order_data = {
    "symbol": "GGEZ1/USDT",
    "price": 0.0875,
    "amount": 100,
    "direction": OrderDirection.BUY.value,
    "type": OrderType.LIMIT_PRICE.value,
}
result = v1_get("/api/order/add", order_data)
pp(result)

### V1 Cancel Order

In [ ]:
order_id = "2521810953500080664"
result = v1_get(f"/api/order/cancel/{order_id}")
pp(result)

### V1 Active Orders

In [ ]:
active = v1_get("/api/order/current/now", {"symbol": "GGEZ1/USDT"})
pp(active)

### V1 Order History

In [ ]:
history = v1_get("/api/order/history", {"symbol": "GGEZ1/USDT", "pageNo": 1, "pageSize": 100})
pp(history)

### V1 Order Info

In [ ]:
order_id = "E17442686601238904"
order_info = v1_get(f"/api/order/detail/{order_id}")
pp(order_info)

### V1 Market Data (Tickers)

In [ ]:
tickers = v1_get("/api/symbol-thumb")
pp(tickers)

### V1 Spot Depth

In [ ]:
depth = v1_get("/api/exchange-plate-depth", {"symbol": "GGEZ1/USDT", "depth": 20})
pp(depth)

### V1 Spot Klines

In [ ]:
time_to = int(time.time() * 1000)
time_from = time_to - (7 * 24 * 60 * 60 * 1000)
klines = v1_get("/api/history", {
    "symbol": "GGEZ1/USDT",
    "from": time_from,
    "to": time_to,
    "resolution": "1D",
})
pp(klines)

### V1 Contract Data

In [ ]:
# Contract symbols
contract_symbols = v1_get_public("/api/v1/swap/symbols")
pp(contract_symbols)

In [ ]:
# Contract order book
contract_depth = api_get_public("/notification/swap/BTCUSDT/orderbook", {"interval": "step0"})
pp(contract_depth)

---
## WebSocket Test

In [ ]:
import asyncio
import websockets


async def listen_orderbook():
    uri = "wss://stream.uzx.com/notification/ws"
    async with websockets.connect(uri) as ws:
        print(f"Connected to {uri}")

        messages = [
            {
                "event": "sub",
                "params": {
                    "biz": "spot",
                    "type": "spot.orderBook",
                    "symbol": "GGEZ1-USDT",
                    "interval": "0",
                },
                "zip": False,
            },
        ]

        for msg in messages:
            await ws.send(json.dumps(msg))
            print(f"-> Sent: {msg}")
            await asyncio.sleep(2)

        print("\n-- Listening for responses --\n")
        try:
            async for message in ws:
                print(f"<- {message}")
        except websockets.ConnectionClosed:
            print("Connection closed by server")


try:
    asyncio.run(listen_orderbook())
except KeyboardInterrupt:
    print("Interrupted by user")